In [0]:
import re
import pandas as pd
from pathlib import Path

# Caminho para o arquivo de dados
CAMINHO = str(Path('../data/DataCoSupplyChainDataset.csv').resolve())

colunas = [
    'Shipping Mode',
    'Delivery Status',
    'Late_delivery_risk',
    'Days for shipping (real)',
    'Days for shipment (scheduled)',
    'Customer City',
    'Order Region'
]
tabela = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("encoding", "ISO-8859-1")
    .csv(CAMINHO)
)

# Delta não aceita espaços, parênteses e vírgulas nos nomes de coluna
tabela = tabela.toDF(*[re.sub(r'[^0-9a-zA-Z_]+', '_', c).strip('_') for c in tabela.columns])

tabela.write.format("Delta").mode("overwrite").saveAsTable("tabela")

display(tabela)



In [0]:
%sql
SELECT 
    Shipping_Mode,
    SUM(CASE WHEN Delivery_Status = 'Late delivery' THEN 1 ELSE 0 END) AS Atrasado,
    COUNT(Shipping_Mode) AS Total,
    Round((SUM(CASE WHEN Delivery_Status = 'Late delivery' THEN 1 ELSE 0 END) * 100.0 / COUNT(Shipping_Mode)),2) AS `% Atraso`
FROM 
    tabela
WHERE 
    Delivery_Status <> 'Shipping canceled'
GROUP BY 
    Shipping_Mode

ORDER BY `% Atraso` DESC
